# Caso B — Data Augmentation mediante Perturbación con Ruido Natural (Noise & Dropout)

## 1. Fundamento y Objetivo
En una situación vocacional real, casi ningún estudiante humano obtiene una evaluación perfecta de 1.0 (100%) en todas las habilidades requeridas para su carrera predilecta. Sin embargo, un estudiante con alta vocación compartirá la gran mayoría de habilidades requeridas con un desempeño alto o notable (ej. 80%, 85%, 90%).

El **Caso B (Perturbación con Ruido)** simula esta condición humana realista a partir de los perfiles Happy Path de cada carrera aplicando dos perturbaciones estocásticas controladas:
1. **Reducción de Cumplimiento (Score Degradation):** Disminución de `cumplimiento_criterio` de 1.0 a valores entre 0.40 y 0.90 en un porcentaje de sus habilidades.
2. **Omisión de Habilidades (Dropout):** Eliminación aleatoria de 1 a 2 habilidades requeridas del perfil del usuario.

**Resultado:** Genera casos de coincidencia alta e imperfecta (50.0% a 95.0%) centrados en su carrera vocacional original.

### Esquema de Identificación:
Cada usuario generado lleva el prefijo de UUID: `PERT-<uuid>`.

### Celda 1: Importaciones y Conexión

In [1]:
import random
import uuid
from typing import Optional

import polars as pl
import sqlalchemy as sa
from faker import Faker
from IPython.display import display
from pydantic import BaseModel

DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = 'password'
DB_CONECTION = 'athena'

connection_url = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_CONECTION}'
)
engine = sa.create_engine(connection_url)
fake = Faker('es_MX')

### Celda 2: Modelos Pydantic

In [2]:
# ── Modelos Pydantic (Validación de Esquema) ─────────────────────────
class Area(BaseModel):
    id_area: int
    nombre_area: str

class Carrera(BaseModel):
    id_carrera: int
    nombre_carrera: str

class Habilidad(BaseModel):
    id_habilidad: int
    descripcion: str

class Usuario(BaseModel):
    id_usuario: Optional[int] = None
    uuid_usuario: str
    nombre: str
    correo: str
    id_area: Optional[int] = None

class Resultado(BaseModel):
    id_resultado: Optional[int] = None
    uuid_usuario: str
    id_carrera: int
    porcentaje_coincidencia: float = 1.0
    top: int = 1

### Celda 3: Helpers de Base de Datos

In [3]:
# ── Funciones Helper de Consulta y Persistencia en BD ──────────────────
def get_all_areas() -> list[dict]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_area, nombre_area
            FROM area
            ORDER BY id_area;
        ''')).mappings().all()
    return [
        {'id_area': row['id_area'], 'nombre_area': row['nombre_area']}
        for row in rows
    ]


def get_all_careers() -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            ORDER BY id_carrera;
        ''')).mappings().all()
    return [
        Carrera(
            id_carrera=row['id_carrera'],
            nombre_carrera=row['nombre_carrera']
        )
        for row in rows
    ]


def get_careers_by_area_select(id_area: int) -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            WHERE id_area = :id_area
            ORDER BY id_carrera;
        '''), {'id_area': id_area}).mappings().all()
    return [
        Carrera(
            id_carrera=row['id_carrera'],
            nombre_carrera=row['nombre_carrera']
        )
        for row in rows
    ]


def get_habilities_by_career(id_carrera: int) -> list[Habilidad]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT h.id_habilidad, h.descripcion
            FROM carrera_habilidad ch
            JOIN habilidad h ON h.id_habilidad = ch.id_habilidad
            WHERE ch.id_carrera = :id_carrera
            ORDER BY h.id_habilidad;
        '''), {'id_carrera': id_carrera}).mappings().all()
    return [
        Habilidad(
            id_habilidad=row['id_habilidad'],
            descripcion=row['descripcion']
        )
        for row in rows
    ]


def submit_user(user: Usuario) -> Usuario:
    with engine.begin() as conn:
        row = conn.execute(sa.text('''
            INSERT INTO usuario (nombre, correo, uuid_usuario, id_area)
            VALUES (:nombre, :correo, :uuid_usuario, :id_area)
            RETURNING id_usuario, uuid_usuario;
        '''), {
            'nombre': user.nombre,
            'correo': user.correo,
            'uuid_usuario': user.uuid_usuario,
            'id_area': user.id_area
        }).mappings().one()
    return Usuario(
        id_usuario=row['id_usuario'],
        uuid_usuario=str(row['uuid_usuario']),
        nombre=user.nombre,
        correo=user.correo,
        id_area=user.id_area,
    )


def construct_result(
    uuid_usuario: str,
    id_carrera: int,
    porcentaje_coincidencia: float = 1.0,
    top: int = 1
) -> Resultado:
    with engine.begin() as conn:
        conn.execute(sa.text('''
            INSERT INTO resultado
                (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
            VALUES (:uuid_usuario, :id_carrera, :porcentaje_coincidencia, :top)
            ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                top = EXCLUDED.top;
        '''), {
            'uuid_usuario': uuid_usuario,
            'id_carrera': id_carrera,
            'porcentaje_coincidencia': porcentaje_coincidencia,
            'top': top,
        })
    return Resultado(
        uuid_usuario=uuid_usuario,
        id_carrera=id_carrera,
        porcentaje_coincidencia=porcentaje_coincidencia,
        top=top,
    )


def get_happy_path_users_by_area(id_area: int) -> list[dict]:
    '''
    Extrae los usuarios Happy Path originales del area,
    agrupando sus habilidades evaluadas y cumplimiento real.
    '''
    query = '''
        SELECT
            u.id_usuario,
            u.uuid_usuario,
            u.nombre,
            u.correo,
            u.id_area,
            uh.id_habilidad,
            MAX(uh.cumplimiento_criterio) AS cumplimiento_criterio
        FROM usuario u
        JOIN usuario_habilidad uh ON uh.id_usuario = u.id_usuario
        WHERE u.id_area = :id_area
          AND u.uuid_usuario LIKE 'USER-%'
        GROUP BY u.id_usuario, u.uuid_usuario, u.nombre, u.correo,
                 u.id_area, uh.id_habilidad
        ORDER BY u.id_usuario, uh.id_habilidad;
    '''
    with engine.begin() as conn:
        rows = conn.execute(
            sa.text(query),
            {'id_area': id_area}
        ).mappings().all()
    usuarios = {}
    for row in rows:
        uid = row['id_usuario']
        if uid not in usuarios:
            usuarios[uid] = {
                'id_usuario': uid,
                'uuid_usuario': row['uuid_usuario'],
                'nombre': row['nombre'],
                'correo': row['correo'],
                'id_area': row['id_area'],
                'habilidades': {}
            }
        usuarios[uid]['habilidades'][row['id_habilidad']] = float(
            row['cumplimiento_criterio']
        )
    return list(usuarios.values())

In [4]:
def generate_name_and_email() -> tuple[str, str]:
    name = fake.name()
    fake_email = fake.email()
    email_local, email_domain = fake_email.split('@', 1)
    email = f'{email_local}_{uuid.uuid4().hex[:10]}@{email_domain}'
    return name, email


_uncached_get_habilities_by_career = get_habilities_by_career
_career_skills_cache = {}


def get_habilities_by_career(id_carrera: int) -> list[Habilidad]:
    if id_carrera not in _career_skills_cache:
        _career_skills_cache[id_carrera] = _uncached_get_habilities_by_career(id_carrera)
    return _career_skills_cache[id_carrera]

### Celda 4: Parámetros y Algoritmo de Perturbación
Configura las tasas de perturbación y aplica la degradación estocástica controlada.

In [5]:
N_VARIACIONES = 3
PCTG_REDUCCION = 0.40
MIN_CUMPL = 0.40
MAX_CUMPL = 0.90
MIN_DROPOUT = 1
MAX_DROPOUT = 2


def persist_cases(pending: list[dict], resumen: list[dict]) -> None:
    if not pending:
        return
    with engine.begin() as conn:
        conn.execute(sa.text('''
            INSERT INTO usuario (nombre, correo, uuid_usuario, id_area)
            VALUES (:nombre, :correo, :uuid_usuario, :id_area);
        '''), [
            {
                'nombre': item['nombre'],
                'correo': item['correo'],
                'uuid_usuario': item['uuid_usuario'],
                'id_area': item['id_area']
            }
            for item in pending
        ])
        inserted = conn.execute(sa.text('''
            SELECT id_usuario, uuid_usuario
            FROM usuario
            WHERE uuid_usuario = ANY(:uuids);
        '''), {'uuids': [item['uuid_usuario'] for item in pending]}).mappings().all()
        ids_by_uuid = {row['uuid_usuario']: row['id_usuario'] for row in inserted}
        for item in pending:
            item['id_usuario'] = ids_by_uuid[item['uuid_usuario']]
            resumen[item['summary_index']]['id_usuario'] = item['id_usuario']

        skills = [
            {
                'id_usuario': item['id_usuario'],
                'id_habilidad': skill_id,
                'cumplimiento_criterio': value
            }
            for item in pending
            for skill_id, value in item['habilidades'].items()
        ]
        conn.execute(sa.text('''
            INSERT INTO usuario_habilidad
                (id_usuario, id_habilidad, cumplimiento_criterio)
            VALUES (:id_usuario, :id_habilidad, :cumplimiento_criterio);
        '''), skills)
        results = [
            {
                'uuid_usuario': item['uuid_usuario'],
                'id_carrera': item['id_carrera'],
                'porcentaje_coincidencia': item['porcentaje_coincidencia'],
                'top': 1
            }
            for item in pending
        ]
        conn.execute(sa.text('''
            INSERT INTO resultado
                (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
            VALUES (:uuid_usuario, :id_carrera, :porcentaje_coincidencia, :top)
            ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                top = EXCLUDED.top;
        '''), results)


def run_perturbation(persist_to_db: bool = True) -> list[dict]:
    resumen = []
    pending = []
    career_cache = {}
    for area in get_all_areas():
        id_area = area['id_area']
        for usuario_hp in get_happy_path_users_by_area(id_area):
            original = dict(usuario_hp['habilidades'])
            with engine.begin() as conn:
                row = conn.execute(sa.text('''
                    SELECT id_carrera
                    FROM resultado
                    WHERE uuid_usuario = :uuid
                    LIMIT 1;
                '''), {'uuid': usuario_hp['uuid_usuario']}).mappings().first()
            if row is None:
                continue

            career_id = row['id_carrera']
            if career_id not in career_cache:
                career_cache[career_id] = get_habilities_by_career(career_id)
            career_skills = career_cache[career_id]
            required_ids = {skill.id_habilidad for skill in career_skills}
            total_required = len(required_ids)

            for variation in range(N_VARIACIONES):
                varied = dict(original)
                reduce_count = max(1, int(len(varied) * PCTG_REDUCCION))
                for skill_id in random.sample(
                    list(varied), k=min(reduce_count, len(varied))
                ):
                    varied[skill_id] = round(
                        random.uniform(MIN_CUMPL, MAX_CUMPL), 2
                    )
                dropout_count = random.randint(
                    MIN_DROPOUT,
                    min(MAX_DROPOUT, max(1, len(varied) - 1))
                )
                if len(varied) > 1:
                    for skill_id in random.sample(
                        list(varied), k=dropout_count
                    ):
                        del varied[skill_id]

                common_ids = required_ids.intersection(varied)
                completion_sum = sum(varied[skill_id] for skill_id in common_ids)
                percentage = (
                    round(completion_sum / total_required, 4)
                    if total_required else 0.0
                )
                generated_uuid = f'PERT-{uuid.uuid4()}'
                name, email = generate_name_and_email()
                summary_index = len(resumen)
                resumen.append({
                    'id_usuario': None,
                    'uuid_usuario': generated_uuid,
                    'variacion': variation + 1,
                    'id_area': id_area,
                    'id_carrera': career_id,
                    'total_requeridas': total_required,
                    'habilidades_tras_dropout': len(varied),
                    'suma_cumplimiento': round(completion_sum, 2),
                    'porcentaje_coincidencia': percentage
                })
                pending.append({
                    'summary_index': summary_index,
                    'uuid_usuario': generated_uuid,
                    'nombre': name,
                    'correo': email,
                    'id_area': id_area,
                    'id_carrera': career_id,
                    'habilidades': varied,
                    'porcentaje_coincidencia': percentage
                })

    by_area = {}
    for item in pending:
        by_area.setdefault(item['id_area'], []).append(item)
    if len(by_area) < 3:
        raise ValueError('No hay candidatos en las tres areas.')
    target = min(len(items) for items in by_area.values())
    balanced_pending = []
    balanced_resumen = []
    for area_id in sorted(by_area):
        for item in by_area[area_id][:target]:
            old_index = item['summary_index']
            item['summary_index'] = len(balanced_resumen)
            balanced_resumen.append(resumen[old_index])
            balanced_pending.append(item)
    pending, resumen = balanced_pending, balanced_resumen

    if persist_to_db:
        persist_cases(pending, resumen)
    else:
        for index, item in enumerate(resumen, start=1):
            item['id_usuario'] = f'SIM-P{index}'
    return resumen


resumen_pert = run_perturbation(persist_to_db=True)
df_pert = pl.DataFrame(resumen_pert)
print(f'Casos Caso B insertados: {df_pert.height}')
print(df_pert.group_by('id_area').len().sort('id_area'))
print(f'Rango: [{df_pert["porcentaje_coincidencia"].min():.4f}, {df_pert["porcentaje_coincidencia"].max():.4f}]')
display(df_pert.head(6))

Casos Caso B insertados: 99
shape: (3, 2)
┌─────────┬─────┐
│ id_area ┆ len │
│ ---     ┆ --- │
│ i64     ┆ u32 │
╞═════════╪═════╡
│ 1       ┆ 33  │
│ 2       ┆ 33  │
│ 3       ┆ 33  │
└─────────┴─────┘
Rango: [0.4100, 0.7967]


id_usuario,uuid_usuario,variacion,id_area,id_carrera,total_requeridas,habilidades_tras_dropout,suma_cumplimiento,porcentaje_coincidencia
i64,str,i64,i64,i64,i64,i64,f64,f64
397,"""PERT-7a79c658-1c6d-49e5-ba7e-4…",1,1,1,7,5,4.48,0.64
398,"""PERT-cc45e3ad-28e7-4eb0-997c-7…",2,1,1,7,5,4.43,0.6329
399,"""PERT-96dec264-2e01-4a1d-b984-a…",3,1,1,7,5,5.0,0.7143
400,"""PERT-59f73bb2-10dd-413e-bf34-2…",1,1,2,6,5,4.05,0.675
401,"""PERT-dcb4c1c4-73f6-4a64-8d78-4…",2,1,2,6,5,4.26,0.71
402,"""PERT-7c508c8e-ba37-451d-bce6-b…",3,1,2,6,5,4.43,0.7383


### Celda 5: Distribución de Deciles para Caso B

In [6]:
dist_b = df_pert.with_columns(
    ((pl.col('porcentaje_coincidencia') * 10).floor() / 10.0).alias('decil')
).group_by('decil').len().sort('decil')

print('=== DISTRIBUCIÓN PORCENTUAL CASO B (ALTOS IMPERFECTOS) ===')
total_b = df_pert.height
for row in dist_b.iter_rows(named=True):
    pct = (row['len'] / total_b) * 100
    barra = '█' * int(pct // 2)
    print(f"{row['decil']:.1f} - {row['decil']+0.1:.1f}: {row['len']:3d} casos ({pct:5.1f}%) | {barra}")

=== DISTRIBUCIÓN PORCENTUAL CASO B (ALTOS IMPERFECTOS) ===
0.4 - 0.5:  12 casos ( 12.1%) | ██████
0.5 - 0.6:  25 casos ( 25.3%) | ████████████
0.6 - 0.7:  32 casos ( 32.3%) | ████████████████
0.7 - 0.8:  30 casos ( 30.3%) | ███████████████


In [7]:
# Verificacion de cobertura de las tres areas en Caso B.
expected_area_ids = {area['id_area'] for area in get_all_areas()}
if len(expected_area_ids) != 3:
    raise ValueError(
        f'Se esperaban 3 areas configuradas y se encontraron {len(expected_area_ids)}.'
    )

actual_area_ids = set(df_pert['id_area'].unique().to_list())
missing_area_ids = expected_area_ids - actual_area_ids
if missing_area_ids:
    raise ValueError(
        f'Caso B no genero usuarios en las areas: {sorted(missing_area_ids)}'
    )

print('Cobertura Caso B por area:')
print(df_pert.group_by('id_area').len().sort('id_area'))

Cobertura Caso B por area:
shape: (3, 2)
┌─────────┬─────┐
│ id_area ┆ len │
│ ---     ┆ --- │
│ i64     ┆ u32 │
╞═════════╪═════╡
│ 1       ┆ 33  │
│ 2       ┆ 33  │
│ 3       ┆ 33  │
└─────────┴─────┘
